In [7]:
import numpy as np
import pandas as pd

from pathlib import Path
import sys

project_root = Path(r"C:\Users\rajur\OneDrive\Desktop\next-move-ai")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.feature_engineering import *

In [8]:
cleaned_path = (
    project_root / "data" / "processed" / "main_questions_cleaned.csv"
)

split_path = (
    project_root / "data" / "processed" / "split_assignments.csv"
)

main_questions = pd.read_csv(cleaned_path)
split_assignments = pd.read_csv(split_path)

print("Main questions:", main_questions.shape)
print("Split assignments:", split_assignments.shape)

print("\nMain question columns:")
print(main_questions.columns.tolist())

print("\nSplit assignment distribution:")
print(split_assignments["partition"].value_counts())

Main questions: (275458, 8)
Split assignments: (4217, 3)

Main question columns:
['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count', 'answer_type']

Split assignment distribution:
partition
train_pool    3373
test           844
Name: count, dtype: int64


In [9]:
fold_datasets = []

for fold in range(1, 6):

    # Identify training and validation students for this fold
    train_users = split_assignments[
        (split_assignments["partition"] == "train_pool") &
        (split_assignments["validation_fold"] != fold)
    ]["user_id"]

    val_users = split_assignments[
        (split_assignments["partition"] == "train_pool") &
        (split_assignments["validation_fold"] == fold)
    ]["user_id"]

    # Extract their interactions
    fold_train_df = main_questions[
        main_questions["user_id"].isin(train_users)
    ].copy()

    fold_val_df = main_questions[
        main_questions["user_id"].isin(val_users)
    ].copy()

    fold_datasets.append((fold_train_df, fold_val_df))

    print(
        f"Fold {fold}: "
        f"Train students = {len(train_users)}, "
        f"Validation students = {len(val_users)}, "
        f"Train interactions = {len(fold_train_df)}, "
        f"Validation interactions = {len(fold_val_df)}"
    )

Fold 1: Train students = 2698, Validation students = 675, Train interactions = 177324, Validation interactions = 42659
Fold 2: Train students = 2698, Validation students = 675, Train interactions = 176273, Validation interactions = 43710
Fold 3: Train students = 2698, Validation students = 675, Train interactions = 170186, Validation interactions = 49797
Fold 4: Train students = 2699, Validation students = 674, Train interactions = 178311, Validation interactions = 41672
Fold 5: Train students = 2699, Validation students = 674, Train interactions = 177838, Validation interactions = 42145


In [10]:
import importlib
import src.feature_engineering as fe

importlib.reload(fe)

add_question_difficulty_features = fe.add_question_difficulty_features

In [11]:
test_history = pd.DataFrame({
    "user_id":    ["B", "C", "A", "D", "E"],
    "problem_id": [100, 100, 100, 100, 100],
    "order_id":   [100, 200, 300, 400, 600],
    "correct":    [1, 0, 1, 0, 1]
})

test_target = pd.DataFrame({
    "user_id":    ["A"],
    "problem_id": [100],
    "order_id":   [500],
    "correct":    [1]
})

test_result = add_question_difficulty_features(
    data=test_target,
    prior_data=test_history,
    alpha=10
)

print(test_result.T)

                                           0
user_id                                    A
problem_id                               100
order_id                                 500
correct                                    1
question_previous_interactions           3.0
question_previous_correct                1.0
question_global_prior                    0.5
question_smoothed_accuracy          0.461538
log_question_previous_interactions  1.386294


In [12]:
validation_target = pd.DataFrame({
    "user_id": ["V"],
    "problem_id": [100],
    "order_id": [500],
    "correct": [1]
})

training_history = pd.DataFrame({
    "user_id": ["B", "C", "D"],
    "problem_id": [100, 100, 100],
    "order_id": [100, 200, 400],
    "correct": [1, 0, 0]
})

validation_result = add_question_difficulty_features(
    data=validation_target,
    prior_data=training_history,
    alpha=10
)

print(validation_result.T)

                                           0
user_id                                    V
problem_id                               100
order_id                                 500
correct                                    1
question_previous_interactions           3.0
question_previous_correct                1.0
question_global_prior               0.333333
question_smoothed_accuracy          0.333333
log_question_previous_interactions  1.386294


In [13]:
import importlib
import src.feature_engineering as fe

importlib.reload(fe)

add_question_difficulty_features = fe.add_question_difficulty_features

test_result = add_question_difficulty_features(
    data=test_target,
    prior_data=test_history,
    alpha=10
)

print(test_result.T)

                                           0
user_id                                    A
problem_id                               100
order_id                                 500
correct                                    1
question_previous_interactions           3.0
question_previous_correct                1.0
question_global_prior                    0.5
question_smoothed_accuracy          0.461538
log_question_previous_interactions  1.386294


In [14]:
result_A = add_question_difficulty_features(
    data=test_target,
    prior_data=test_history,
    alpha=10
)

print(result_A[
    [
        "question_previous_interactions",
        "question_previous_correct",
        "question_global_prior",
        "question_smoothed_accuracy"
    ]
])

   question_previous_interactions  question_previous_correct  \
0                             3.0                        1.0   

   question_global_prior  question_smoothed_accuracy  
0                    0.5                    0.461538  


In [15]:
fold_train_df, fold_val_df = fold_datasets[0]

fold1_targets = pd.concat(
    [
        fold_train_df.assign(_partition="train"),
        fold_val_df.assign(_partition="validation")
    ],
    ignore_index=True
)

print("Combined targets:", fold1_targets.shape)
print("Training targets:", (fold1_targets["_partition"] == "train").sum())
print("Validation targets:", (fold1_targets["_partition"] == "validation").sum())

Combined targets: (219983, 9)
Training targets: 177324
Validation targets: 42659


In [16]:
import time

start_time = time.perf_counter()

fold1_features = add_question_difficulty_features(
    data=fold1_targets,
    prior_data=fold_train_df,
    alpha=10
)

elapsed_time = time.perf_counter() - start_time

print(f"Feature generation completed in {elapsed_time:.2f} seconds")
print("Generated shape:", fold1_features.shape)

Feature generation completed in 41.63 seconds
Generated shape: (219983, 14)


In [17]:
fold1_train_features = (
    fold1_features[
        fold1_features["_partition"] == "train"
    ]
    .drop(columns="_partition")
    .reset_index(drop=True)
)

fold1_val_features = (
    fold1_features[
        fold1_features["_partition"] == "validation"
    ]
    .drop(columns="_partition")
    .reset_index(drop=True)
)

feature_columns = [
    "question_previous_interactions",
    "question_previous_correct",
    "question_global_prior",
    "question_smoothed_accuracy",
    "log_question_previous_interactions"
]

print("Training features:", fold1_train_features.shape)
print("Validation features:", fold1_val_features.shape)

print("\nTraining feature summary:")
print(fold1_train_features[feature_columns].describe().round(4))

print("\nValidation feature summary:")
print(fold1_val_features[feature_columns].describe().round(4))

Training features: (177324, 13)
Validation features: (42659, 13)

Training feature summary:
       question_previous_interactions  question_previous_correct  \
count                     177324.0000                177324.0000   
mean                          11.9383                     7.6385   
std                           14.4551                     9.2801   
min                            0.0000                     0.0000   
25%                            2.0000                     1.0000   
50%                            7.0000                     4.0000   
75%                           16.0000                    10.0000   
max                          107.0000                    87.0000   

       question_global_prior  question_smoothed_accuracy  \
count            177324.0000                 177324.0000   
mean                  0.6656                      0.6632   
std                   0.0225                      0.0972   
min                   0.5000                      0.167

In [18]:
print("TRAINING SUMMARY")
print(
    fold1_train_features[feature_columns]
    .describe()
    .round(4)
)

print("\nVALIDATION SUMMARY")
print(
    fold1_val_features[feature_columns]
    .describe()
    .round(4)
)

TRAINING SUMMARY
       question_previous_interactions  question_previous_correct  \
count                     177324.0000                177324.0000   
mean                          11.9383                     7.6385   
std                           14.4551                     9.2801   
min                            0.0000                     0.0000   
25%                            2.0000                     1.0000   
50%                            7.0000                     4.0000   
75%                           16.0000                    10.0000   
max                          107.0000                    87.0000   

       question_global_prior  question_smoothed_accuracy  \
count            177324.0000                 177324.0000   
mean                  0.6656                      0.6632   
std                   0.0225                      0.0972   
min                   0.5000                      0.1679   
25%                   0.6533                      0.6073   
50%       

In [19]:
for name, df in [
    ("Training", fold1_train_features),
    ("Validation", fold1_val_features)
]:
    print(f"\n{name} checks")

    print(
        "Missing values:",
        df[feature_columns].isna().sum().sum()
    )

    print(
        "Negative interaction counts:",
        (df["question_previous_interactions"] < 0).sum()
    )

    print(
        "Correct count exceeds total:",
        (
            df["question_previous_correct"]
            > df["question_previous_interactions"]
        ).sum()
    )

    print(
        "Accuracy outside [0, 1]:",
        (
            (df["question_smoothed_accuracy"] < 0) |
            (df["question_smoothed_accuracy"] > 1)
        ).sum()
    )


Training checks
Missing values: 0
Negative interaction counts: 0
Correct count exceeds total: 0
Accuracy outside [0, 1]: 0

Validation checks
Missing values: 0
Negative interaction counts: 0
Correct count exceeds total: 0
Accuracy outside [0, 1]: 0


In [20]:
feature_output_dir = (
    project_root
    / "data"
    / "processed"
    / "engineered_features"
    / "group_4_question_difficulty"
    / "base_features"
)

feature_output_dir.mkdir(parents=True, exist_ok=True)

fold1_train_features.to_csv(
    feature_output_dir / "fold_1_train.csv",
    index=False
)

fold1_val_features.to_csv(
    feature_output_dir / "fold_1_validation.csv",
    index=False
)

print("Fold 1 features saved successfully!")

Fold 1 features saved successfully!


In [21]:
for fold_number in range(2, 6):

    print(f"\nGenerating Fold {fold_number}...")

    fold_train_df, fold_val_df = fold_datasets[fold_number - 1]

    fold_targets = pd.concat(
        [
            fold_train_df.assign(_partition="train"),
            fold_val_df.assign(_partition="validation")
        ],
        ignore_index=True
    )

    start_time = time.perf_counter()

    fold_features = add_question_difficulty_features(
        data=fold_targets,
        prior_data=fold_train_df,
        alpha=10
    )

    elapsed_time = time.perf_counter() - start_time

    train_features = (
        fold_features[fold_features["_partition"] == "train"]
        .drop(columns="_partition")
        .reset_index(drop=True)
    )

    val_features = (
        fold_features[fold_features["_partition"] == "validation"]
        .drop(columns="_partition")
        .reset_index(drop=True)
    )

    train_features.to_csv(
        feature_output_dir / f"fold_{fold_number}_train.csv",
        index=False
    )

    val_features.to_csv(
        feature_output_dir / f"fold_{fold_number}_validation.csv",
        index=False
    )

    print("Training shape:", train_features.shape)
    print("Validation shape:", val_features.shape)
    print(f"Execution time: {elapsed_time:.2f} seconds")
    print("Saved successfully.")


Generating Fold 2...
Training shape: (176273, 13)
Validation shape: (43710, 13)
Execution time: 79.31 seconds
Saved successfully.

Generating Fold 3...
Training shape: (170186, 13)
Validation shape: (49797, 13)
Execution time: 113.57 seconds
Saved successfully.

Generating Fold 4...
Training shape: (178311, 13)
Validation shape: (41672, 13)
Execution time: 124.92 seconds
Saved successfully.

Generating Fold 5...
Training shape: (177838, 13)
Validation shape: (42145, 13)
Execution time: 69.32 seconds
Saved successfully.


In [22]:
saved_files = sorted(feature_output_dir.glob("*.csv"))

print("Number of saved files:", len(saved_files))

for file in saved_files:
    df = pd.read_csv(file)

    print(
        file.name,
        "| Shape:", df.shape,
        "| Missing:", df.isna().sum().sum()
    )

Number of saved files: 10
fold_1_train.csv | Shape: (177324, 13) | Missing: 11163
fold_1_validation.csv | Shape: (42659, 13) | Missing: 2286
fold_2_train.csv | Shape: (176273, 13) | Missing: 10270
fold_2_validation.csv | Shape: (43710, 13) | Missing: 3179
fold_3_train.csv | Shape: (170186, 13) | Missing: 10570
fold_3_validation.csv | Shape: (49797, 13) | Missing: 2879
fold_4_train.csv | Shape: (178311, 13) | Missing: 11012
fold_4_validation.csv | Shape: (41672, 13) | Missing: 2437
fold_5_train.csv | Shape: (177838, 13) | Missing: 10781
fold_5_validation.csv | Shape: (42145, 13) | Missing: 2668


In [23]:
fold1_train = pd.read_csv(
    feature_output_dir / "fold_1_train.csv"
)

fold1_val = pd.read_csv(
    feature_output_dir / "fold_1_validation.csv"
)

print("TRAINING MISSING VALUES")
print(
    fold1_train.isna().sum()[
        fold1_train.isna().sum() > 0
    ]
)

print("\nVALIDATION MISSING VALUES")
print(
    fold1_val.isna().sum()[
        fold1_val.isna().sum() > 0
    ]
)

TRAINING MISSING VALUES
skill_id    11163
dtype: int64

VALIDATION MISSING VALUES
skill_id    2286
dtype: int64


In [24]:
feature_columns = [
    "question_previous_interactions",
    "question_previous_correct",
    "question_global_prior",
    "question_smoothed_accuracy",
    "log_question_previous_interactions"
]

print("Training engineered-feature missing values:")
print(fold1_train[feature_columns].isna().sum())

print("\nValidation engineered-feature missing values:")
print(fold1_val[feature_columns].isna().sum())

Training engineered-feature missing values:
question_previous_interactions        0
question_previous_correct             0
question_global_prior                 0
question_smoothed_accuracy            0
log_question_previous_interactions    0
dtype: int64

Validation engineered-feature missing values:
question_previous_interactions        0
question_previous_correct             0
question_global_prior                 0
question_smoothed_accuracy            0
log_question_previous_interactions    0
dtype: int64


In [25]:
print("Original cleaned dataset:")
print("Missing skill_id:", main_questions["skill_id"].isna().sum())

print("\nFold 1 training:")
print("Missing skill_id:", fold1_train_features["skill_id"].isna().sum())

print("\nFold 1 validation:")
print("Missing skill_id:", fold1_val_features["skill_id"].isna().sum())

Original cleaned dataset:
Missing skill_id: 16059

Fold 1 training:
Missing skill_id: 11163

Fold 1 validation:
Missing skill_id: 2286
